In [ ]:
import xarray as xr
import rioxarray as rxr
from pyproj import Transformer
import numpy as np
import glob
import re
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import retrieve_url, str2time
from data_funcs import int2fstep

## Functions

TODO: Move to subdirectory

In [ ]:
band_df_hrrr = pd.DataFrame({
    'Band': [616, 620, 624, 628, 629, 661, 561, 612, 643],
    'hrrr_name': ['TMP', 'RH', "WIND", 'PRATE', 'APCP',
                  'DSWRF', 'SOILW', 'CNWAT', 'GFLUX'],
    'dict_name': ["temp", "rh", "wind", "rain", "precip_accum",
                 "solar", "soilm", "canopyw", "groundflux"],
    'descr': ['2m Temperature [K]', 
              '2m Relative Humidity [%]', 
              '10m Wind Speed [m/s]'
              'surface Precip. Rate [kg/m^2/s]',
              'surface Total Precipitation [kg/m^2]',
              'surface Downward Short-Wave Radiation Flux [W/m^2]',
              'surface Total Precipitation [kg/m^2]',
              '0.0m below ground Volumetric Soil Moisture Content [Fraction]',
              'Plant Canopy Surface Water [kg/m^2]',
              'surface Ground Heat Flux [W/m^2]']
})

In [ ]:
def bands_to_names(bands):
    # Get the bands from band list, assumes band_df_hrrr exists in memory
    # Find matching dict_name values in the dataframe
    dict_names = band_df_hrrr.loc[band_df_hrrr['Band'].isin(bands), 'dict_name'].tolist()
    return dict_names

In [ ]:
bands_to_names([561, 612])

In [ ]:
def names_to_bands(names):
    # Get the names from file names, assumes band_df_hrrr exists in memory
    # Find matching band values in the dataframe
    bands = band_df_hrrr.loc[band_df_hrrr['dict_name'].isin(names), 'Band'].tolist()
    return bands  

In [ ]:
names_to_bands(['temp', 'rh'])

In [ ]:
def calc_eqs(ds):

    # Calculate Ed based on temp and rh
    temp = ds.sel(band="temp")
    rh = ds.sel(band="rh")
    
    Ed = 0.924 * rh**0.679 + 0.000499 * np.exp(0.1 * rh) + 0.18 * (21.1 + 273.15 - temp) * (1 - np.exp(-0.115 * rh))
    Ew = 0.618 * rh**0.753 + 0.000454 * np.exp(0.1 * rh) + 0.18 * (21.1 + 273.15 - temp) * (1 - np.exp(-0.115 * rh))
    
    # Expand dims and assign new band names for Ed and Ew
    Ed = Ed.expand_dims(dim="band").assign_coords(band=["Ed"])
    Ew = Ew.expand_dims(dim="band").assign_coords(band=["Ew"])

    ds = xr.concat([ds, Ed, Ew], dim="band")
    
    return ds
    

In [ ]:
def extract_timestamp(file_path):
    # Extract date (parent directory) and hour from the file path
    date_str = re.search(r'(\d{8})', file_path).group(1)  # Matches YYYYMMDD
    hour_str = re.search(r't(\d{2})z', file_path).group(1)  # Matches tHHz
    
    # Combine into a datetime object
    timestamp = datetime.strptime(f"{date_str} {hour_str}", "%Y%m%d %H")
    return timestamp

In [ ]:
# Preprocess function to extract time information from filename and set it as a coordinate
def preprocess(ds, add_xy = True):
    # Extract time and assign as coord
    time = extract_timestamp(ds.encoding['source'])
    ds = ds.assign_coords(time=time)  # Add time coordinate
    # Extract band name and assign as coord
    band_number = int(re.search(r'\.(\d{3})\.', ds.encoding['source']).group(1))
    band_name = bands_to_names([band_number])
    ds = ds.assign_coords(band = ("band", band_name))
    
    return ds

In [ ]:
def bbox_to_xy(bbox, crs, epsg = 4326):
    transformer = Transformer.from_crs(f"EPSG:{epsg}", crs, always_xy=True)
    # Transform the lat/lon bounding box to x/y
    minx, miny = transformer.transform(bbox[1], bbox[0])  # (min_lon, min_lat)
    maxx, maxy = transformer.transform(bbox[3], bbox[2])  # (max_lon, max_lat)

    return minx, miny, maxx, maxy

In [ ]:
bands = [616, 620, 628]
# hrs = ["00", "01", "02"] # Hour of day 00-23
fstep = int2fstep(0)
doy_str = "20240101"

In [ ]:
def get_file_list(start_time, end_time, fstep, bands_list, base_path = "."):
    # Set up time
    t0 = datetime.strptime(str(start_time), "%Y%m%d%H")
    t1 = datetime.strptime(str(end_time), "%Y%m%d%H")  
    assert t1 > t0, "end_time must be after start_time"
    times = pd.date_range(start=t0,end=t1, freq="1H")

    # Generate File list based on saved HRRR band format
    file_list = []
    for time in times:
        doy_str = time.strftime("%Y%m%d")
        hr = time.strftime("%H")
        files = [f"{doy_str}/hrrr.t{hr}z.wrfprs{fstep}.{band}.tif" for band in bands_list]
        file_list.append(files) # NOTE: appending to make list nested for data reading with xarray
    
    return file_list

In [ ]:
start_time = 2024042000
end_time = 2024042001
forecast_step = 3
base_url = "https://demo.openwfm.org/web/data/fmda/tif/"

In [ ]:
fstep = int2fstep(forecast_step)
if forecast_step > 0:
    fprev = int2fstep(forecast_step-1)
print(f"{fstep=}")
print(f"{fprev=}")

In [ ]:
file_list = get_file_list(start_time, end_time, fstep=fstep, bands_list = [629])
file_list

In [ ]:
# Rain file list, used to calculate hourly rainfall
file_list_prev = get_file_list(start_time, end_time, fstep=fprev, bands_list = [629])
file_list_prev

## Retrieve Data Remotely

In [ ]:
for sublist in file_list:
    for file in sublist:
        retrieve_url(
            f"{base_url}/{file}",
            dest_path = f"{file}"
        )

In [ ]:
for sublist in file_list_prev:
    for file in sublist:
        print(file)
        retrieve_url(
            f"{base_url}/{file}",
            dest_path = f"{file}"
        )

## Read Grouped Data and Calculate

In [ ]:
data = xr.open_mfdataset(
    file_list,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

data.attrs["forecast_step"] = fstep

data_prev = xr.open_mfdataset(
    file_list_prev,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

data_prev.attrs["forecast_step"] = fprev

In [ ]:
# data = calc_eqs(data)

In [ ]:
data.time.shape

In [ ]:
data.band

In [ ]:
data.dims

In [ ]:
data.band

In [ ]:
data_prev.band

In [ ]:
data.band_data.shape

In [ ]:
data.band_data.shape

In [ ]:
data_prev.band_data.shape

In [ ]:
data.dims

In [ ]:
data_prev.dims

In [ ]:
def calc_rain(ds, ds_prev):
    # Check times are the same
    assert np.all(data.time.values == data_prev.time.values), "Time dimension not the same between input xarrays"
    
    rain = ds.sel(band="precip_accum") - ds_prev.sel(band="precip_accum")
    rain = rain.expand_dims(dim="band").assign_coords(band=["rain"])

    ds = xr.concat([ds, rain], dim="band")
    return ds

In [ ]:
data = calc_rain(data, data_prev)

In [ ]:
data.band

In [ ]:
data.band_data.shape

In [ ]:
data.time[1]

In [ ]:
plt.imshow(data.sel(band = "rain", time = '2024-04-20T01:00:00.000000000')['band_data'])

## Subsetting to bbox

In [ ]:
def get_projection_info(ds, epsg = 4326):
    # Given a geotiff file (a HRRR band), 
    # return info necessary to transform lat/lon coords to the file structure
    # Inputs: 
    # ds: (osgeo.gdal.Dataset)
    # epsg: (int) default 4326 for lon/lat
    # Return: (tuple) with fields (ct, g_inv)
        # ct: (osgeo.osr.CoordinateTransformation)
        # gt_inv: (tuple) output of gdal.InvGeoTransform, also could be found with gdalinfo on command line
    gt = ds.GetGeoTransform()
    gp = ds.GetProjection()
    if(ds.RasterCount>1):
        print('Not Implemented for multiple Raster bands')
        sys.exit(-1)
    # Get Projection info
    point_srs = osr.SpatialReference()
    point_srs.ImportFromEPSG(4326) # hardcode for lon/lat
    # GDAL>=3: make sure it's x/y
    # see https://trac.osgeo.org/gdal/wiki/rfc73_proj6_wkt2_srsbarn
    point_srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
    file_srs = osr.SpatialReference()
    file_srs.ImportFromWkt(gp)
    ct = osr.CoordinateTransformation(point_srs, file_srs)
    gt_inv = gdal.InvGeoTransform(gt)

    return ct, gt_inv

In [ ]:
type(data)

In [ ]:
ds2 = rxr.open_rasterio(file_list[0])

In [ ]:
ds2.rio.transform()

In [ ]:
data.rio.crs

In [ ]:
ds2

In [ ]:
plt.imshow(ds2.isel(band=0))

In [ ]:
bbox = [37, -111, 46, -95]
crs = data.rio.crs

In [ ]:
def bbox_to_xy(bbox, crs, epsg = 4326):
    transformer = Transformer.from_crs(f"EPSG:{epsg}", crs, always_xy=True)
    # Transform the lat/lon bounding box to x/y
    minx, miny = transformer.transform(bbox[1], bbox[0])  # (min_lon, min_lat)
    maxx, maxy = transformer.transform(bbox[3], bbox[2])  # (max_lon, max_lat)

    return minx, miny, maxx, maxy

In [ ]:
minx, miny, maxx, maxy = bbox_to_xy(bbox, crs)

In [ ]:
ds2_clipped = ds2.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)

In [ ]:
plt.imshow(ds2_clipped.isel(band=0))

In [ ]:
ds2_clipped.coords

In [ ]:
features_list

In [ ]:
data_clipped = data2.sel(x=slice(minx, maxx), y=slice(maxy, miny), band=features_list)  # Note: flip y for descending order

In [ ]:
data_clipped

In [ ]:
data_clipped.band_data.shape

In [ ]:
data_clipped.dims

In [ ]:
X = data_clipped.band_data.values

In [ ]:
X.shape

In [ ]:
type(X)

In [ ]:
X.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
scaler

In [ ]:
data2.band

In [ ]:
data3 = data2.sel(band = features_list)

In [ ]:
data3

In [ ]:
from utils import read_pkl
rnn_dat = read_pkl("../outputs/models/rnn_data_rocky.pkl")

In [ ]:
type(rnn_dat)

In [ ]:
rnn_dat.scaler

In [ ]:
rnn_dat.features_list